In [2]:
import tensorflow as tf
import numpy as np
# 先用原始 tf.keras 加载 MNIST（在切换 compat 之前）
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype('float32')  # 图中已有 xs/255. 做归一化
x_test = x_test.reshape(-1, 784).astype('float32')
y_train_onehot = np.eye(10)[y_train.astype('int32')].astype('float32')
y_test_onehot = np.eye(10)[y_test.astype('int32')].astype('float32')
class MNISTData:
    class Train:
        def next_batch(self, n):
            idx = np.random.choice(len(x_train), n, replace=False)
            return x_train[idx], y_train_onehot[idx]
    def __init__(self):
        self.train = self.Train()
        self.test = type('Test', (), {'images': x_test, 'labels': y_test_onehot})()
mnist = MNISTData()
learning_rate = 1e-4
keep_prob_rate = 0.7  # 训练时保留神经元的比例
max_epoch = 2000

def weight_variable(shape):
    initial = tf.random.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    # 每一维度  滑动步长全部是 1， padding 方式 选择 same
    # 提示 使用函数  tf.nn.conv2d
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    # 滑动步长 是 2步; 池化窗口的尺度 高和宽度都是2; padding 方式 请选择 same
    # 提示 使用函数  tf.nn.max_pool
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

# 卷积层 1 + 卷积层 2 + 全连接（与作业要求相同的 8 处结构）
W_conv1 = weight_variable([7, 7, 1, 32])   # patch 7x7, in size 1, out size 32
b_conv1 = bias_variable([32])
W_conv2 = weight_variable([5, 5, 32, 64])  # patch 5x5, in size 32, out size 64
b_conv2 = bias_variable([64])
W_fc1 = weight_variable([7*7*64, 1024])
b_fc1 = bias_variable([1024])
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])

trainable_vars = [W_conv1, b_conv1, W_conv2, b_conv2, W_fc1, b_fc1, W_fc2, b_fc2]

def forward(x, training=True):
    x_image = tf.reshape(x / 255.0, [-1, 28, 28, 1])
    # 卷积层 1
    h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
    h_pool1 = max_pool_2x2(h_conv1)
    # 卷积层 2
    h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
    h_pool2 = max_pool_2x2(h_conv2)
    # 全连接
    h_pool2_flat = tf.reshape(h_pool2, [-1, 7*7*64])
    h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
    h_fc1_drop = tf.nn.dropout(h_fc1, rate=1 - keep_prob_rate) if training else h_fc1
    return tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)

optimizer = tf.keras.optimizers.Adam(learning_rate)

for i in range(max_epoch):
    batch_xs, batch_ys = mnist.train.next_batch(100)
    bx = tf.constant(batch_xs, dtype=tf.float32)
    by = tf.constant(batch_ys, dtype=tf.float32)
    with tf.GradientTape() as tape:
        pred = forward(bx, training=True)
        loss = tf.reduce_mean(-tf.reduce_sum(by * tf.math.log(pred + 1e-10), axis=1))
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))
    if i % 100 == 0:
        pred_test = forward(mnist.test.images[:1000], training=False)
        acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(pred_test, 1), tf.argmax(mnist.test.labels[:1000], 1)), tf.float32))
        print(float(acc))


0.07199999690055847
0.8820000290870667
0.9190000295639038
0.9290000200271606
0.9399999976158142
0.9490000009536743
0.9480000138282776
0.9570000171661377
0.968999981880188
0.9639999866485596
0.968999981880188
0.9729999899864197
0.9729999899864197
0.9750000238418579
0.9729999899864197
0.9769999980926514
0.9789999723434448
0.9789999723434448
0.9789999723434448
0.9779999852180481
